# **Section 3: Revision - Booleans, conditionals, loops and simulation - Part 1**

Everything here runs on its own, the data is built
in the notebook, so you can change a number and re-run to see what happens.

**Chapters 9 and 10.1 of the textbook.**

**Where this is used.** Foundational. Loops, conditionals and simulation are the
shape of the bootstrap in Section 5 and of the classifier loops in Section 7, but no
project asks for them directly.


In [ ]:
# Run this cell first
%pip install -q datascience ipywidgets

try:
    import pyodide_http
    pyodide_http.patch_all()
except ImportError:
    pass

from datascience import *
import numpy as np
%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')
import warnings
warnings.simplefilter('ignore', FutureWarning)

---

## **Contents**

1. [Comparisons produce booleans](#1)
2. [`=` and `==` are different](#2)
3. [True counts as 1](#3)
4. [`if` statements](#4)
5. [`for` loops](#5)
6. [Growing an array with `np.append`](#6)
7. [Probability by hand](#7)
8. [Simulation: the recipe](#8)
9. [A worked simulation](#9)
10. [**>>Quick reference<<**](#10)
11. [**>>Self-check<<**](#11)
12. [Quick reference](#12)


---

<a id='1'></a>
## **1. Comparisons produce booleans**

A comparison is a question. The answer is `True` or `False`, a value of type
`bool`, which you can store, count and compute with.


In [ ]:
x = 3
y = 4

print(x == y)      # is x equal to y?
print(x != y)      # is x different from y?
print(x < y)
print(y >= x - 1)

In [ ]:
# Comparisons can be chained -- this asks BOTH questions at once
print(2 < x < 5)
print(type(x < y))

Comparisons work element by element on an array, giving an array of booleans, 
one answer per element.


In [ ]:
names = make_array('Thandi', 'Sipho', 'Thandi', 'Naledi')
names == 'Thandi' 

In [ ]:
np.arange(5) > 2

---

<a id='2'></a>
## **2. `=` and `==` are different**

`=` **assigns**. It changes what a name refers to, and returns nothing.

`==` **asks**. It compares two values and gives back `True` or `False`.

Mixing them up is the single most common error in this chapter.


In [ ]:
x = 3
y = 4

x == y      # asks a question -- x is untouched

In [ ]:
x = y       # ASSIGNS -- x now refers to 4
print('x is', x)
print('y is', y)

> **Watch out.** After `x = y`, the old value of `x` is gone. There is no
> warning, and the cell produces no output, which is exactly why the mistake
> is easy to miss.


---

<a id='3'></a>
## **3. True counts as 1**

Python treats `True` as 1 and `False` as 0. That makes counting easy:

| To find | Use |
|---|---|
| How many are True | `sum(array == value)` or `np.count_nonzero(...)` |
| What proportion are True | `np.average(array == value)` |


In [ ]:
names = make_array('Thandi', 'Sipho', 'Thandi', 'Naledi')

print(names == 'Thandi')
print('count      :', sum(names == 'Thandi'))
print('count again:', np.count_nonzero(names == 'Thandi'))
print('proportion :', np.average(names == 'Thandi'))

`np.count_nonzero` is the clearer of the two, because it says what it does.
`sum` works because `True` is 1, a useful trick, but less readable.


In [ ]:
# A proportion is just a mean of 0s and 1s
scores = make_array(45, 72, 88, 51, 96, 63)
print('how many above 60 :', np.count_nonzero(scores > 60))
print('proportion above  :', np.average(scores > 60))

---

<a id='4'></a>
## **4. `if` statements**

```
if <condition>:
    <do this>
elif <another condition>:
    <do this instead>
else:
    <otherwise do this>
```

Python checks the conditions **top to bottom and stops at the first True one**.
Everything below it is skipped, even if it would also have been true.


In [ ]:
def sign(x):
    """Returns 'Positive', 'Negative' or 'Neither'."""
    if x > 0:
        return 'Positive'
    elif x < 0:
        return 'Negative'
    else:
        return 'Neither'

print(sign(3))
print(sign(-3))
print(sign(0))

### **Order matters**

The two functions below look similar and behave differently. In the second one,
the first condition catches everything, so the later branches never run.


In [ ]:
def grade_right(score):
    if score >= 75:
        return 'Distinction'
    elif score >= 50:
        return 'Pass'
    else:
        return 'Fail'

def grade_wrong(score):
    if score >= 50:          # catches 80 as well
        return 'Pass'
    elif score >= 75:
        return 'Distinction'  # unreachable
    else:
        return 'Fail'

print('right:', grade_right(80))
print('wrong:', grade_wrong(80))

> **Common mistake.** A function that runs off the end without hitting a
> `return` gives back `None`, silently. `sign('-1')` above would compare a
> string to a number and raise an error, try it.


---

<a id='5'></a>
## **5. `for` loops**

```
for <name> in <array>:
    <do something with name>
```

The body runs once per element. The name takes each value in turn.


In [ ]:
for i in np.arange(5):
    print('i is', i)

In [ ]:
faculty = make_array('Thandi', 'Sipho', 'Naledi')

for person in faculty:
    print('Hello,', person)

`np.arange(n)` is how you repeat something `n` times. The loop variable is
often unused, it is just a counter.


In [ ]:
# A running total
total = 0
for k in np.arange(1, 6):
    total = total + k
    print('after adding', k, 'the total is', total)

---

<a id='6'></a>
## **6. Growing an array with `np.append`**

Simulation almost always follows the same shape: start with an empty array, run
a trial, append the result, repeat.

```
results = make_array()
for i in np.arange(repetitions):
    outcome = <one trial>
    results = np.append(results, outcome)
```

The reassignment matters. `np.append` returns a **new** array; it does not
change the old one.


In [ ]:
rolls = make_array()

for i in np.arange(5):
    one_roll = np.random.choice(np.arange(1, 7))
    rolls = np.append(rolls, one_roll)

rolls

> **Common mistake.** Writing `np.append(results, outcome)` without the
> `results =` in front. The new array is built and thrown away, and `results`
> stays empty, with no error to tell you.


---

<a id='7'></a>
## **7. Probability by hand**

Three rules cover almost everything in this chapter.

| Rule | When | Formula |
|---|---|---|
| Multiplication | all of several things happen | multiply the chances |
| Complement | "at least one" | 1 - P(none) |
| Addition | one of several **separate** outcomes | add the chances |

A roulette wheel has 38 pockets: 18 red, 18 black, 2 green.


In [ ]:
# All of the first three spins are black
first_three_black = (18/38) ** 3
first_three_black

In [ ]:
# Green never wins in 10 spins -- 36 of the 38 pockets are not green
no_green = (36/38) ** 10
no_green

In [ ]:
# Green wins AT LEAST ONCE in 10 spins.
# Do not add up the ways it could happen -- take the complement.
at_least_one_green = 1 - (36/38) ** 10
at_least_one_green

> **The "at least one" trap.** Students often try `10 * (2/38)`, adding the
> chance for each spin. That double-counts the outcomes where green wins more
> than once, and for a large enough number of spins it gives an answer above 1.
> Always go through the complement.


In [ ]:
# Two of the three colours never win in 10 spins:
# that means ONE colour wins every time. Three separate ways -> add them.
lone_winners = (18/38)**10 + (18/38)**10 + (2/38)**10
lone_winners

---

<a id='8'></a>
## **8. Simulation: the recipe**

When the arithmetic is hard, simulate. Every simulation in this course has the
same four parts:

1. **One trial**, a function that plays the game once and returns the outcome
2. **An empty array** to collect results
3. **A loop** that runs the trial many times and appends each outcome
4. **A summary**, count, average or histogram the collected results

Getting part 1 right is most of the work. If you can write a function that does
one trial, the rest is the same every time.


In [ ]:
# 1. one trial
def one_roll_is_six():
    """Rolls one die, returns True if it lands on 6."""
    return np.random.choice(np.arange(1, 7)) == 6

one_roll_is_six()

In [ ]:
# 2, 3 and 4
outcomes = make_array()

for i in np.arange(10000):
    outcomes = np.append(outcomes, one_roll_is_six())

print('simulated:', np.average(outcomes))
print('exact    :', 1/6)

---

<a id='9'></a>
## **9. A worked simulation**

**Question.** You spin a roulette wheel 10 times. What is the chance green wins
at least once?

We already know the exact answer from Section 7. Simulating it is a way to check
that the simulation is set up correctly, always worth doing when you can.


In [ ]:
wheel_colours = np.append(np.append(np.repeat('red', 18),
                                    np.repeat('black', 18)),
                          np.repeat('green', 2))
print(len(wheel_colours), 'pockets')
print(np.count_nonzero(wheel_colours == 'green'), 'green')

In [ ]:
def green_in_ten_spins():
    """Spins ten times, returns True if green came up at least once."""
    ten_spins = np.random.choice(wheel_colours, 10)
    return np.count_nonzero(ten_spins == 'green') > 0

green_in_ten_spins()

In [ ]:
results = make_array()

for i in np.arange(10000):
    results = np.append(results, green_in_ten_spins())

print('simulated:', np.average(results))
print('exact    :', 1 - (36/38)**10)

Close, but not identical, and it will differ slightly every time you run it.
That is the point: a simulated answer is an **estimate**, and its accuracy
improves with more repetitions. Change 10000 to 100 and run it again.


---

<a id='10'></a>
## **10. Quick questions**

Five of them. Set `my_answer` to a letter and run the cell.
A wrong answer gets a nudge so you can try again; a right one gets the reason.

These come before the written questions below on purpose: they are quicker,
and they check the things people most often get wrong.

In [ ]:
# Run this once. mcq.py must be in the same folder as this notebook.
from mcq import check_answer, show_answer

**M1.** Why does this leave `results` empty?

```python
results = make_array()
for i in np.arange(5):
    np.append(results, i)
```

**a)** `make_array()` cannot be extended  
**b)** the result of `np.append` is never assigned  
**c)** `np.arange(5)` is empty  
**d)** you cannot append inside a loop  

In [ ]:
my_answer = '?'          # a, b, c or d
check_answer('w3_m1', my_answer)

**M2.** `ages` holds 100 values. What does `ages == 30` produce?

**a)** an array of 100 booleans  
**b)** a single `True` or `False`  
**c)** the number of ages equal to 30  
**d)** an array of the ages equal to 30  

In [ ]:
my_answer = '?'          # a, b, c or d
check_answer('w3_m2', my_answer)

**M3.** Which gives the **proportion** of ages equal to 30?

**a)** `np.count_nonzero(ages == 30)`  
**b)** `sum(ages == 30)`  
**c)** `np.average(ages == 30)`  
**d)** `len(ages == 30)`  

In [ ]:
my_answer = '?'          # a, b, c or d
check_answer('w3_m3', my_answer)

**M4.** An `if` has three branches and two conditions are true. Which branch runs?

**a)** the first true one  
**b)** the last true one  
**c)** both of them  
**d)** the `else`  

In [ ]:
my_answer = '?'          # a, b, c or d
check_answer('w3_m4', my_answer)

**M5.** What is the difference between `=` and `==`?

**a)** none, they are interchangeable  
**b)** `==` assigns, `=` compares  
**c)** `=` is for numbers, `==` for text  
**d)** `=` assigns and returns nothing; `==` compares and gives `True` or `False`  

In [ ]:
my_answer = '?'          # a, b, c or d
check_answer('w3_m5', my_answer)

---

<a id='11'></a>
## **11. Self-check**

Answer these without scrolling back. Reveal each answer only after you have committed to one.

**Q1.** What is the difference between `x = 5` and `x == 5`, and which can stand alone as a complete statement?

<details>
<summary><strong>Answer</strong></summary>

<code>=</code> <strong>assigns</strong>: it puts 5 into the name <code>x</code>, and is a complete statement. <code>==</code> <strong>asks</strong> whether <code>x</code> equals 5 and produces <code>True</code> or <code>False</code>. Writing <code>==</code> where you meant <code>=</code> gives a value nobody uses; the reverse is usually a syntax error.

</details>

**Q2.** Why does this loop leave `results` empty?

```python
results = make_array()
for i in np.arange(5):
    np.append(results, i)
```

<details>
<summary><strong>Answer</strong></summary>

<code>np.append</code> does not change the array it is given. It returns a <strong>new</strong> array, and here the return value is thrown away. The line must be <code>results = np.append(results, i)</code>. Nothing warns you, so the symptom is an empty array at the end.

</details>

**Q3.** `np.count_nonzero(ages == 30)` and `np.average(ages == 30)` use the same comparison. What does each give?

<details>
<summary><strong>Answer</strong></summary>

<code>count_nonzero</code> gives the <strong>number</strong> equal to 30. <code>average</code> gives the <strong>proportion</strong>, because <code>True</code> counts as 1 and <code>False</code> as 0, so the mean of the booleans is the fraction that are True.

</details>

**Q4.** An `if` has three branches and two of the conditions are true. Which branch runs?

<details>
<summary><strong>Answer</strong></summary>

The <strong>first</strong> one whose condition is true. The rest are never tested. This is why order matters: a broad condition placed first swallows every case a narrower one below it was meant to catch.

</details>

**Q5.** What is the difference between `np.random.choice(a, 5)` and `np.random.choice(a, 5, replace=False)`?

<details>
<summary><strong>Answer</strong></summary>

The first draws <strong>with</strong> replacement, so the same element can appear more than once. The second draws <strong>without</strong> replacement, so every draw is distinct and <code>a</code> must hold at least 5 elements.

</details>

**Q6.** What does `ages == 30` produce when `ages` holds 100 values?

<details>
<summary><strong>Answer</strong></summary>

An <strong>array of 100 booleans</strong>, one per element, not a single True or False. That is what makes counting and averaging work on it. If you wanted one answer, you may have meant <code>np.all</code> or <code>np.any</code>.

</details>

---

<a id='12'></a>
## **12. Quick reference**

### **Booleans**

| Call | Gives |
|---|---|
| `x == y` | True if equal |
| `x != y` | True if different |
| `2 < x < 5` | both comparisons at once |
| `array == value` | an array of booleans, one per element |
| `np.count_nonzero(array == value)` | how many are True |
| `np.average(array == value)` | what proportion are True |

### **Control flow**

| Call | Does |
|---|---|
| `if` / `elif` / `else` | runs the first branch whose condition is True |
| `for name in array:` | runs the body once per element |
| `np.arange(n)` | the counter for "do this n times" |
| `np.append(arr, value)` | returns a NEW array with value added |

### **Randomness**

| Call | Gives |
|---|---|
| `np.random.choice(array)` | one element, at random |
| `np.random.choice(array, n)` | n elements, **with** replacement |
| `np.random.choice(array, n, replace=False)` | n elements, without replacement |

### **Probability**

| Situation | Do |
|---|---|
| All of several things happen | multiply |
| At least one happens | 1 − P(none) |
| One of several separate outcomes | add |

### **Things that catch people out**

| | |
|---|---|
| `=` vs `==` | assigns vs asks |
| `np.append` without reassigning | the result is discarded, silently |
| "At least one" by adding | double-counts; use the complement |
| `if` order | the first True branch wins; later ones never run |
| A function with no `return` | gives back `None` |

---

### **Where this comes from in the textbook**

- [Chapter 9, Randomness](https://inferentialthinking.com/chapters/09/randomness/)
- [Chapter 9.1, Conditional statements](https://inferentialthinking.com/chapters/09/1/conditional-statements/)
- [Chapter 9.2, Iteration](https://inferentialthinking.com/chapters/09/2/iteration/)
- [Chapter 9.5, Finding probabilities](https://inferentialthinking.com/chapters/09/5/finding-probabilities/)
